<a href="https://colab.research.google.com/github/ayush-614/APSLAB614/blob/main/IBMinternship.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Securing your NEWS_API_KEY

It's a good practice to store sensitive information like API keys using Colab's secrets manager. You can access it by clicking the "🔑" icon on the left panel. Add your `NEWS_API_KEY` there. Once added, you can load it into your notebook like this:

In [1]:
!pip install flask flask-cors pyngrok pandas scikit-learn nltk requests

In [2]:
import pandas as pd
import pickle
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier

corpus = {
    'text': [
        "Official minutes from the university council confirm standard academic grading criteria remain unchanged.",
        "A verified atmospheric anomaly was tracked across the northern hemisphere by automated satellite arrays.",
        "The department finalized infrastructure updates to support expanding campus computing requirements.",
        "Shocking hidden leak reveals entire city water systems are transmitting mind control micro-substances.",
        "Unbelievable dynamic matrix allows anyone to manifest physical gold bars out of thin air overnight.",
        "Medical breakthrough confirmed by independent peer-reviewed laboratory testing platforms globally."
    ],
    'label': [1, 1, 1, 0, 0, 1]
}
df = pd.DataFrame(corpus)

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.15, random_state=42)

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.8)
X_train_tfidf = vectorizer.fit_transform(X_train)

classifier = PassiveAggressiveClassifier(max_iter=100)
classifier.fit(X_train_tfidf, y_train)

with open('model.pkl', 'wb') as m_file:
    pickle.dump(classifier, m_file)

with open('vectorizer.pkl', 'wb') as v_file:
    pickle.dump(vectorizer, v_file)

In [3]:
import os
import pickle
import requests
import re
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

NEWS_API_KEY = "1d5ba9a0753f4e23be110b0bbe6372df"
NGROK_AUTH_TOKEN = "3FAdiuprxd9ZHjjvPzN0ItjTAmB_67yk13KStznwkcBMggvnm"

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

with open('model.pkl', 'rb') as m_file:
    model = pickle.load(m_file)

with open('vectorizer.pkl', 'rb') as v_file:
    vectorizer = pickle.load(v_file)

def extract_intelligent_keywords(text):
    stop_words = {
        'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', 'arent', 'as', 'at',
        'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'cant', 'cannot',
        'could', 'couldnt', 'did', 'didnt', 'do', 'does', 'doesnt', 'doing', 'dont', 'down', 'during', 'each', 'few',
        'for', 'from', 'further', 'had', 'hadnt', 'has', 'hasnt', 'have', 'havent', 'having', 'he', 'hed', 'hell',
        'hes', 'her', 'here', 'heres', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'hows', 'i', 'id', 'ill',
        'im', 'ive', 'if', 'in', 'into', 'is', 'isnt', 'it', 'its', 'itself', 'lets', 'me', 'more', 'most', 'mustnt',
        'my', 'myself', 'no', 'nor', 'not', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'ought', 'our', 'ours',
        'ourselves', 'out', 'over', 'own', 'same', 'shant', 'she', 'shed', 'shell', 'shes', 'should', 'shouldnt', 'so',
        'some', 'such', 'than', 'that', 'thats', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there',
        'theres', 'these', 'they', 'theyd', 'theyll', 'theyre', 'theyve', 'this', 'those', 'through', 'to', 'too',
        'under', 'until', 'up', 'very', 'was', 'wasnt', 'we', 'wed', 'well', 'were', 'weve', 'werent', 'what', 'whats',
        'when', 'whens', 'where', 'wheres', 'which', 'while', 'who', 'whos', 'whom', 'why', 'whys', 'with', 'wont',
        'would', 'wouldnt', 'you', 'youd', 'youll', 'youre', 'youve', 'your', 'yours', 'yourself', 'yourselves'
    }
    cleaned_text = re.sub(r'[^\w\s]', '', text.lower())
    words = cleaned_text.split()
    filtered_keywords = [w for w in words if w not in stop_words and len(w) > 2]
    return filtered_keywords

def fetch_external_validation(query_text):
    if not query_text.strip():
        return False, "Insufficient textual token distribution for cross-reference extraction."

    keywords = extract_intelligent_keywords(query_text)
    if not keywords:
        return False, "Could not synthesize strong subject anchors from the provided text layer."

    search_query = " ".join(keywords[:3])
    target_url = f"https://newsapi.org/v2/everything?q={search_query}&apiKey={NEWS_API_KEY}&pageSize=5"

    try:
        response = requests.get(target_url, timeout=5)
        if response.status_code == 200:
            articles = response.json().get('articles', [])
            valid_sources = []

            critical_matches = [k for k in keywords if len(k) > 3]

            for a in articles:
                title_lower = a['title'].lower() if a['title'] else ""
                desc_lower = a['description'].lower() if a['description'] else ""

                match_count = sum(1 for word in critical_matches if word in title_lower or word in desc_lower)

                if match_count >= 2:
                    valid_sources.append(f"[{a['source']['name']}] {a['title']}")

            if len(valid_sources) >= 2:
                return True, "Active topic alignment found across mainstream news media:\n\n" + "\n".join(valid_sources[:3])

            return False, f"Discrepancy detected: No reputable news sources are confirming a story with these specific terms."

        return False, "External API validation layer returned an operational exception."
    except Exception:
        return False, "External API validation layer encountered a connection timeout."

@app.route('/predict', methods=['POST'])
def process_prediction():
    try:
        payload = request.get_json()
        if not payload or 'news_content' not in payload:
            return jsonify({'error': 'Invalid payload signature. Missing news_content key.'}), 400

        raw_text = payload['news_content']
        transformed_vector = vectorizer.transform([raw_text])
        numeric_prediction = model.predict(transformed_vector)[0]

        web_confirmed, web_report = fetch_external_validation(raw_text)

        if numeric_prediction == 1 and web_confirmed:
            pred_text = "Verified / Credible Analytical Alignment"
            res_class = "real"
            summary_text = "The structural linguistic patterns within this text display high statistical alignment with standard verified journalism networks, and the live web registry successfully confirmed the event context."
        else:
            pred_text = "Unverified / Discrepancy Flagged"
            res_class = "fake"
            summary_text = "Linguistic pattern analysis anomalies or a lack of supporting live media documentation indicates high structural convergence with unverified information networks."

        return jsonify({
            'prediction': 0 if not web_confirmed else int(numeric_prediction),
            'prediction_text': pred_text,
            'result_class': res_class,
            'summary_text': summary_text,
            'web_verification': web_report
        })
    except Exception as error:
        return jsonify({'error': f"Internal processing pipeline exception: {str(error)}"}), 500

if __name__ == '__main__':
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(5000)
    print("=" * 60)
    print(f"TARGET DEPLOYMENT ENDPOINT: {tunnel.public_url}")
    print("=" * 60)
    app.run(port=5000, debug=False, use_reloader=False)

TARGET DEPLOYMENT ENDPOINT: https://shrubbery-busload-disown.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:20:55] "OPTIONS /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:20:56] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:23:52] "OPTIONS /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:23:53] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:24:17] "OPTIONS /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:24:18] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:25:26] "OPTIONS /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:25:27] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12:27:48] "OPTIONS /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jun/2026 12